# 01 - Prepare data (run with **Accelerator: None**, internet ON)

Downloads TinyStories-V2, trains the BPE tokenizer, tokenizes into `train.bin`/`val.bin`.
Doing this on a CPU session avoids burning GPU quota. When it finishes, **Save Version -> Save & Run All (Commit)**; the output (`/kaggle/working/tinygpt_data`) is what the training notebook attaches.

Before running: upload `tinygpt.zip` as a Kaggle Dataset (Datasets -> New Dataset) and click **Add Data** to attach it.

In [ ]:
import glob, os, shutil, subprocess, sys

def sh(cmd):
    """Run a shell command and stream its output live; raise if it fails."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

def setup_code():
    """Copy the tinygpt code (uploaded as a Kaggle Dataset) into /kaggle/working and install it."""
    work = "/kaggle/working/tinygpt"
    if not os.path.exists(work):
        hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
        assert hits, "Code not found. Upload tinygpt.zip as a Kaggle Dataset and attach it (Add Data)."
        shutil.copytree(os.path.dirname(hits[0]), work)
    os.chdir(work)
    sh("pip install -q -e . regex pyyaml matplotlib")
    return work

setup_code()

In [ ]:
sh('python -m pytest -q tests/test_tokenizer.py tests/test_data.py')   # quick sanity check of the code

In [ ]:
sh('python -m tinygpt.data.download --out data/raw')

In [ ]:
sh('python -m tinygpt.tokenizer.train --input data/raw/TinyStoriesV2-GPT4-train.txt --max-bytes 200000000 --vocab-size 4096 --out data/tok/tokenizer.json')

In [ ]:
# optional: compare compression with Hugging Face tokenizers (target: within 2%)
sh('pip install -q tokenizers && python scripts/tokenizer_parity.py --train data/raw/TinyStoriesV2-GPT4-train.txt --valid data/raw/TinyStoriesV2-GPT4-valid.txt --tok data/tok/tokenizer.json')

In [ ]:
sh(f'python -m tinygpt.data.prepare --tok data/tok/tokenizer.json --train data/raw/TinyStoriesV2-GPT4-train.txt --valid data/raw/TinyStoriesV2-GPT4-valid.txt --out data/shards --workers {max(1, os.cpu_count() - 1)}')

In [ ]:
os.makedirs('/kaggle/working/tinygpt_data', exist_ok=True)
sh('cp -r data/tok data/shards /kaggle/working/tinygpt_data/')
sh('rm -rf data/raw')      # raw text is not needed any more (keeps the output small)
sh('cat /kaggle/working/tinygpt_data/shards/meta.json && du -sh /kaggle/working/tinygpt_data')